In [1]:
import casadi
import random
import cvxpy as cp
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time
from pytope import Polytope
from matplotlib.patches import Ellipse
import matplotlib.animation as animation

from Initial_Set_Estimation import Initial_Set_Estimation
from ModelingSVTrue import ModelingSVTrue
from Planner_P import Planner_P
from Planner_D import Planner_D
from Planner_R import Planner_R
from Planner_S import Planner_S

%matplotlib inline

In [2]:
def draw_ellipse(center, matrix):
    # Matrix is Q of (x-q)'*Q^(-1)*(x-1)
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    width, height = 2*np.sqrt(eigenvalues)
    angle   = np.arctan2(*eigenvectors[:,0][::-1])
    angle   = np.degrees(angle)
    ellipse = Ellipse(xy = center, width = width, height = height, angle = angle, lw = 2)
    
    return ellipse

def rota_rect(state, l, w):
    # Note: the first four states must be x, y, heading, velocity
    x       = state[0]
    y       = state[1]
    box     = [(x - l/2, y + w/2), (x + l/2, y + w/2), (x + l/2, y - w/2), (x - l/2, y - w/2)]
    heading = state[2]
    
    box_matrix = np.array(box) - np.repeat(np.array([[x, y]]), len(box), 0)
    theta = -heading
    rota_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                            [np.sin(theta), np.cos(theta)]])

    new = box_matrix.dot(rota_matrix) + np.repeat(np.array([[x, y]]), len(box), 0)
    
    x = [new[0][0], new[1][0], new[2][0], new[3][0], new[0][0]]
    y = [new[0][1], new[1][1], new[2][1], new[3][1], new[0][1]]
    return x, y

In [3]:
track_parameters    = np.load('track_para.npy', allow_pickle = True).item( ) # predefined track
u_turn_ref_spline   = track_parameters['u_turn_ref_spline']
x_track_sv          = u_turn_ref_spline[:, 0].T
y_track_sv          = u_turn_ref_spline[:, 1].T

In [4]:
initial_start_ev = 100
initial_start_sv = 99 

x_EV_initial = np.array([2, -initial_start_ev, np.pi/2, 0])
x_SV_initial = np.array([-initial_start_sv, -3.5, 0, 0])

p_order     = 8
w_lane      = 4 
r_round     = 210
r_circle    = 25
T           = 0.25
v_target    = 50/3.6
N_EV        = 10 
N_SV        = 10
v_target_ev = v_target 
v_target_sv = v_target
infinity    = 1000000 # infinity
l_veh       = 4.3 # vehicle length, m
w_veh       = 1.8 # vehicle width, m
DEV         = 4   # dimenstion of EV
DSV         = 4   # dimenstion of SV
num_points_sv = len(x_track_sv)

In [5]:
A_SV      = np.array([[1, T, 0, 0], [0, 1, 0, 0], [0, 0, 1, T], [0, 0, 0, 1]])
B_SV      = np.array([[0.5*T**2, 0], [T, 0], [0, 0.5*T**2], [0, T]])
min_ay    = -7
max_ay    = 7
min_ax    = -7
max_ax    = 7
U_SV_Poly = Polytope([[0, min_ay], [max_ax, 0], [0, max_ay], [min_ax, 0]])
q_robust  = np.zeros((2, 1))
Q_robust  = np.diag([7**2, 7**2])
H         = U_SV_Poly.A
h         = U_SV_Poly.b
nu        = 2
nx        = 4
d_min     = 2*np.sqrt((l_veh/2)**2 + (w_veh/2)**2)

In [6]:
N_Car  = 2
margin = 0.8
opts_Parameters_EV = { 
    'T': T,     
    'N': N_EV,
    'l_f': 1.65,
    'l_r': 1.65,
    'DEV': DEV,
    'nu': nu,
    'nx': nx,
    'A_SV': A_SV,
    'B_SV': B_SV,
    'infinity': infinity,
    'Q1': 10, # steering wheel angle
    'Q2': 1,  # longitudinal acc
    'Q3': 1,  # x. pos. error
    'v_low': 0,
    'v_up': 50/3.6, 
    'acc_low': -3.5, 
    'acc_up': 3,    
    'delta_low': -np.pi/4, 
    'delta_up': np.pi/4, 
    'x_track_sv': x_track_sv,
    'y_track_sv': y_track_sv,
    'num_points_sv': num_points_sv,
    'x_con_c': r_round,
    'y_con_c': -r_round,
    'r_in': r_round - w_lane + w_veh/2 + margin,
    'r_out': r_round - w_veh/2 - margin,
    'p': p_order,
    'margin': margin,
    'acc_max': 5,
    'collision_tol': 0.001,
    'state_tol': 0.001,
    'Q_measure': np.diag([0.01**2, 0.01**2, 0.01**2, 0.01**2]), # of (x-q)'*Q^(-1)*(x-q)
    'Q_safedis': np.diag([d_min**2, d_min**2]),                 # of (x-q)'*Q^(-1)*(x-q)
    'du':0.00001,
    'L_State': np.array([1, 1, 1, 1]).reshape(4, 1),
    'L_Occ': np.array([1, 1]).reshape(2, 1),
    'H': H,
    'h': h,
    'q_robust': q_robust,
    'Q_robust': Q_robust}
opts_Parameters_SVTrue = { 
    'T': T,     
    'N': N_SV,
    'l_f': 1.65,
    'l_r': 1.65,
    'v_low': 0,
    'v_up': v_target_sv, 
    'acc_low': -3.25,
    'acc_up': 3.25,   
    'delta_low': -np.pi/4,
    'delta_up': np.pi/4,
    'x_track': x_track_sv,
    'y_track': y_track_sv,
    'v_target': v_target_sv,
    'num_points': num_points_sv,
    'r_circle': r_circle + w_veh/2,
    'Q1': 1, # steering wheel angle
    'Q2': 1, # longitudinal acceleration
    'Q3': 1, # long. velocity error
    'Q4': 1, # x. pos. error
    'Q5': 1, # y. pos. error
    'DSV': 4}

In [7]:
Planner_P  = Planner_P(Params = opts_Parameters_EV)
Planner_D  = Planner_D(Params = opts_Parameters_EV)
Planner_R  = Planner_R(Params = opts_Parameters_EV)
Planner_S  = Planner_S(Params = opts_Parameters_EV)
ModelingSVTrue   = ModelingSVTrue(Params = opts_Parameters_SVTrue)

In [8]:
samples_SV = np.array([[-0.01, -0.01], [0.01, -0.01], [-0.01, 0.01], [0.01, 0.01]])
N_pre_sam  = 4
opts_Parameters_Initial_Set_Estimation = {'N_pre_sam': 4, 'nu': nu}
Initial_Set_Estimation                 = Initial_Set_Estimation(Params = opts_Parameters_Initial_Set_Estimation)
A_initial, b_initial, Q_initial, q_initial = Initial_Set_Estimation.Return(samples_SV)

Error: rescode.err_license_expired(1001): The license has expired.


In [ ]:
K_N = 63
# SV
State_SV       = np.zeros((4, K_N + 1))
Control_SV     = np.zeros((2, K_N))
X_SV_Planning  = list( ) 
State_SV[:, 0] = x_SV_initial

# Define storage for P
State_EV_P      = np.zeros((DEV, K_N + 1))
Control_EV_P    = np.zeros((2, K_N))
J_EV_P          = np.array([None]*(K_N))
X_EV_Planning_P = list( )
A_P             = list( ) 
b_P             = list( )
Qr_Inv_P         = list( ) 
qr_P             = list( )
Qu_P            = list( ) 
qu_P            = list( )
Flag_P          = list( )
# initiate
State_EV_P[:, 0] = x_EV_initial
A_P.append(A_initial)
b_P.append(b_initial)
Qu_P.append(Q_initial)
qu_P.append(q_initial)

# Define storage for D
State_EV_D      = np.zeros((DEV, K_N + 1))
Control_EV_D    = np.zeros((2, K_N))
J_EV_D          = np.array([None]*(K_N))
X_EV_Planning_D = list( )
Qr_Inv_D             = list( ) 
qr_D             = list( )
Flag_D          = list( )
# initiate
State_EV_D[:, 0] = x_EV_initial

# Define storage for R
State_EV_R      = np.zeros((DEV, K_N + 1))
Control_EV_R    = np.zeros((2, K_N))
J_EV_R          = np.array([None]*(K_N))
X_EV_Planning_R = list( )
Qr_Inv_R             = list( ) 
qr_R             = list( )
Flag_R          = list( )
# initiate
State_EV_R[:, 0] = x_EV_initial

# Define storage for SS (scenario method with small scenarios)
State_EV_SS      = np.zeros((DEV, K_N + 1))
Control_EV_SS    = np.zeros((2, K_N))
J_EV_SS          = np.array([None]*(K_N))
X_EV_Planning_SS = list( )
A_SS             = list( ) 
b_SS             = list( )
Qr_Inv_SS         = list( ) 
qr_SS             = list( )
Qu_SS           = list( ) 
qu_SS            = list( )
Flag_SS          = list( )
# initiate
State_EV_SS[:, 0] = x_EV_initial
A_SS.append(A_initial)
b_SS.append(b_initial)
Qu_SS.append(Q_initial)
qu_SS.append(q_initial)

# Define storage for SL (scenario method with large scenarios)
State_EV_SL      = np.zeros((DEV, K_N + 1))
Control_EV_SL    = np.zeros((2, K_N))
J_EV_SL          = np.array([None]*(K_N))
X_EV_Planning_SL = list( )
A_SL             = list( ) 
b_SL             = list( )
Qr_Inv_SL        = list( ) 
qr_SL             = list( )
Qu_SL            = list( ) 
qu_SL            = list( )
Flag_SL          = list( )
# initiate
State_EV_SL[:, 0] = x_EV_initial
A_SL.append(A_initial)
b_SL.append(b_initial)
Qu_SL.append(Q_initial)
qu_SL.append(q_initial)

In [10]:
N_Sample_Small = 15
N_Sample_Large = 150
Time_P = list()
Time_D = list()
Time_R = list()
Time_SS = list()
Time_SL = list()
for k in range(K_N):
    print('step is', k)
    
    u_sv_before = samples_SV[-1]
    u_sv_before = u_sv_before.reshape(2, 1)
    control_SV_k, x_SV_planning_k  = ModelingSVTrue.Return(State_SV[:, k])
    X_SV_Planning.append(x_SV_planning_k)
    Control_SV[:, k]   = control_SV_k
    State_SV[:, k + 1] = x_SV_planning_k[:, 1]
    
    # Planning with P
    start_P = time.perf_counter( )
    U_EV_k, trajectory_planning_EV_k, J_EV_k, A_k, b_k, Qu_k, qu_k, Qr_Inv, qr, flag_k  = Planner_P.Return(State_SV[:, k], State_EV_P[:, k], A_P[-1], b_P[-1], u_sv_before)
    end_P = time.perf_counter( )
    Time_P.append(end_P - start_P)
    Control_EV_P[:, k] = U_EV_k
    X_EV_Planning_P.append(trajectory_planning_EV_k)
    State_EV_P[:, k+1] = trajectory_planning_EV_k[:, 1]
    J_EV_P[k]  = J_EV_k
    A_P.append(A_k)
    b_P.append(b_k)
    Qr_Inv_P.append(Qr_Inv)
    qr_P.append(qr)
    Qu_P.append(Qu_k)
    qu_P.append(qu_k)
    Flag_P.append(flag_k)
    
    # Planning with D
    start_D = time.perf_counter( )
    U_EV_k, trajectory_planning_EV_k, J_EV_k, Qr_Inv, qr, flag_k  = Planner_D.Return(State_SV[:, k], State_EV_D[:, k])
    end_D = time.perf_counter( )
    Time_D.append(end_D - start_D)
    Control_EV_D[:, k] = U_EV_k
    X_EV_Planning_D.append(trajectory_planning_EV_k)
    State_EV_D[:, k+1] = trajectory_planning_EV_k[:, 1]
    J_EV_D[k]  = J_EV_k
    Qr_Inv_D.append(Qr_Inv)
    qr_D.append(qr)
    Flag_D.append(flag_k)

    # Planning with R
    start_R = time.perf_counter( )
    U_EV_k, trajectory_planning_EV_k, J_EV_k, Qr_Inv, qr, flag_k  = Planner_R.Return(State_SV[:, k], State_EV_R[:, k])
    end_R = time.perf_counter( )
    Time_R.append(end_R - start_R)
    Control_EV_R[:, k] = U_EV_k
    X_EV_Planning_R.append(trajectory_planning_EV_k)
    State_EV_R[:, k+1] = trajectory_planning_EV_k[:, 1]
    J_EV_R[k]  = J_EV_k
    Qr_Inv_R.append(Qr_Inv)
    qr_R.append(qr)
    Flag_R.append(flag_k)

    # Planning with SS
    start_SS = time.perf_counter( )
    U_EV_k, trajectory_planning_EV_k, J_EV_k, A_k, b_k, Qu_k, qu_k, Qr_Inv, qr, flag_k = Planner_S.Return(State_SV[:, k], State_EV_SS[:, k], A_SS[-1], b_SS[-1], u_sv_before, N_Sample_Small)
    end_SS = time.perf_counter( )
    Time_SS.append(end_SS - start_SS)
    Control_EV_SS[:, k] = U_EV_k
    X_EV_Planning_SS.append(trajectory_planning_EV_k)
    State_EV_SS[:, k+1] = trajectory_planning_EV_k[:, 1]
    J_EV_SS[k]  = J_EV_k
    A_SS.append(A_k)
    b_SS.append(b_k)
    Qr_Inv_SS.append(Qr_Inv)
    qr_SS.append(qr)
    Qu_SS.append(Qu_k)
    qu_SS.append(qu_k)
    Flag_SS.append(flag_k)

    # Planning with SL
    start_SL = time.perf_counter( )
    U_EV_k, trajectory_planning_EV_k, J_EV_k, A_k, b_k, Qu_k, qu_k, Qr_Inv, qr, flag_k = Planner_S.Return(State_SV[:, k], State_EV_SL[:, k], A_SL[-1], b_SL[-1], u_sv_before, N_Sample_Large)
    end_SL = time.perf_counter( )
    Time_SL.append(end_SL - start_SL)
    Control_EV_SL[:, k] = U_EV_k
    X_EV_Planning_SL.append(trajectory_planning_EV_k)
    State_EV_SL[:, k+1] = trajectory_planning_EV_k[:, 1]
    J_EV_SL[k] = J_EV_k
    A_SL.append(A_k)
    b_SL.append(b_k)
    Qr_Inv_SL.append(Qr_Inv)
    qr_SL.append(qr)
    Qu_SL.append(Qu_k)
    qu_SL.append(qu_k)
    Flag_SL.append(flag_k)
    
    samples_SV = np.vstack((samples_SV, np.array([Control_SV[:, k][1]*np.cos(State_SV[:, k][2]), Control_SV[:, k][1]*np.sin(State_SV[:, k][2])])))

print ('Average excution time of P: ', np.mean(Time_P))
print ('Average excution time of D: ', np.mean(Time_D))
print ('Average excution time of R: ', np.mean(Time_R))
print ('Average excution time of SS: ', np.mean(Time_SS))
print ('Average excution time of SL: ', np.mean(Time_SL))

t = np.arange(0, T*(K_N + 1), T, dtype=float) # total simulation time
if len(State_SV[0]) < len(t):
    t = t[0:len(State_SV[0])]
print('planning done')

step is 0

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

step is 1
step is 2
step is 3
step is 4
step is 5
step is 6
step is 7
step is 8
step is 9
step is 10
step is 11
step is 12
step is 13
step is 14
step is 15
step is 16
step is 17
step is 18
step is 19
step is 20
step is 21
step is 22
step is 23
step is 24
step is 25
step is 26
step is 27
step is 28
step is 29
step is 30
step is 31
step is 32
step is 33
step is 34
step is 35
step is 36
step is 37
step is 38
step is 39
step is 40
step is 41
step is 42
step is 43
step is 44
step is 45
step is 46
step is 47
step is 48
step is 49
step is 50
step is 51
step is 52
step is 53
step is 54
step is 55
step is

In [11]:
color_ev = 'r'
color_sv = 'b'
color_car = [color_ev, color_sv]

In [ ]:
import os
np.save(os.path.join('1_Data_Case_1', 't'), t)
np.save(os.path.join('1_Data_Case_1', 'State_SV'), State_SV)
np.save(os.path.join('1_Data_Case_1', 'Control_SV'), Control_SV)
np.save(os.path.join('1_Data_Case_1', 'X_SV_Planning'), X_SV_Planning)
np.save(os.path.join('1_Data_Case_1', 'samples_SV'), samples_SV)

np.save(os.path.join('1_Data_Case_1', 'State_EV_P'), State_EV_P)
np.save(os.path.join('1_Data_Case_1', 'Control_EV_P'), Control_EV_P)
np.save(os.path.join('1_Data_Case_1', 'J_EV_P'), J_EV_P)
np.save(os.path.join('1_Data_Case_1', 'X_EV_Planning_P'), X_EV_Planning_P)
np.save(os.path.join('1_Data_Case_1', 'Qr_Inv_P'), Qr_Inv_P)
np.save(os.path.join('1_Data_Case_1', 'qr_P'), qr_P)
np.save(os.path.join('1_Data_Case_1', 'Qu_P'), Qu_P)
np.save(os.path.join('1_Data_Case_1', 'qu_P_data'), qu_P)
np.save(os.path.join('1_Data_Case_1', 'Flag_P'), Flag_P)

np.save(os.path.join('1_Data_Case_1', 'State_EV_D'), State_EV_D)
np.save(os.path.join('1_Data_Case_1', 'Control_EV_D'), Control_EV_D)
np.save(os.path.join('1_Data_Case_1', 'J_EV_D'), J_EV_D)
np.save(os.path.join('1_Data_Case_1', 'X_EV_Planning_D'), X_EV_Planning_D)
np.save(os.path.join('1_Data_Case_1', 'Qr_Inv_D'), Qr_Inv_D)
np.save(os.path.join('1_Data_Case_1', 'qr_D'), qr_D)
np.save(os.path.join('1_Data_Case_1', 'Flag_D'), Flag_D)

np.save(os.path.join('1_Data_Case_1', 'State_EV_R'), State_EV_R)
np.save(os.path.join('1_Data_Case_1', 'Control_EV_R'), Control_EV_R)
np.save(os.path.join('1_Data_Case_1', 'J_EV_R'), J_EV_R)
np.save(os.path.join('1_Data_Case_1', 'X_EV_Planning_R'), X_EV_Planning_R)
np.save(os.path.join('1_Data_Case_1', 'Qr_Inv_R'), Qr_Inv_R)
np.save(os.path.join('1_Data_Case_1', 'qr_R'), qr_R)
np.save(os.path.join('1_Data_Case_1', 'Flag_R'), Flag_R)

np.save(os.path.join('1_Data_Case_1', 'State_EV_SS'), State_EV_SS)
np.save(os.path.join('1_Data_Case_1', 'Control_EV_SS'), Control_EV_SS)
np.save(os.path.join('1_Data_Case_1', 'J_EV_SS'), J_EV_SS)
np.save(os.path.join('1_Data_Case_1', 'X_EV_Planning_SS'), X_EV_Planning_SS)
np.save(os.path.join('1_Data_Case_1', 'Qr_Inv_SS'), Qr_Inv_SS)
np.save(os.path.join('1_Data_Case_1', 'qr_SS'), qr_SS)
np.save(os.path.join('1_Data_Case_1', 'Qu_SS'), Qu_SS)
np.save(os.path.join('1_Data_Case_1', 'qu_SS'), qu_SS)
np.save(os.path.join('1_Data_Case_1', 'Flag_SS'), Flag_SS)

np.save(os.path.join('1_Data_Case_1', 'State_EV_SL'), State_EV_SL)
np.save(os.path.join('1_Data_Case_1', 'Control_EV_SL'), Control_EV_SL)
np.save(os.path.join('1_Data_Case_1', 'J_EV_SL'), J_EV_SL)
np.save(os.path.join('1_Data_Case_1', 'X_EV_Planning_SL'), X_EV_Planning_SL)
np.save(os.path.join('1_Data_Case_1', 'Qr_Inv_SL'), Qr_Inv_SL)
np.save(os.path.join('1_Data_Case_1', 'qr_SL'), qr_SL)
np.save(os.path.join('1_Data_Case_1', 'Qu_SL'), Qu_SL)
np.save(os.path.join('1_Data_Case_1', 'qu_SL'), qu_SL)
np.save(os.path.join('1_Data_Case_1', 'Flag_SL'), Flag_SL)